In [10]:
import pandas as pd
import json
import numpy as np
import re
import ast

from pathlib import Path

In [11]:
# Load the dataset
src_path = Path("../data/processed/dataset_clean.csv")
df = pd.read_csv(src_path)

### papersPerYear

In [12]:
# Compute counts per year
counts = df["Year"] \
        .value_counts(dropna=False) \
        .rename_axis("Year") \
        .reset_index(name="Count")

# Sort by year (ascending)
counts = counts.sort_values("Year")

# Save to CSV
out_path = Path("../site/data/papers/papersPerYearData.csv")
counts.to_csv(out_path, index=False)

### papersPerYearStats

In [13]:
# ---- Input CSV (Year,Count) ----
IN_CSV = "../site/data/papers/papersPerYearData.csv"        
OUT_STATS_CSV = "../site/data/papers/papersPerYearStats.csv"

df = pd.read_csv(IN_CSV)
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Count"] = (
    df["Count"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)

# Keep valid years only
df = df.dropna(subset=["Year"]).copy()
df["Year"] = df["Year"].astype(int)

# ---- Stats ----
total = int(df["Count"].sum())
n_years = int(df["Year"].nunique())  # oppure len(df) se hai una riga per anno
avgPerYear = int(round(total / n_years)) if n_years > 0 else 0

peak_idx = df["Count"].idxmax() if len(df) else None
peakYear = int(df.loc[peak_idx, "Year"]) if peak_idx is not None else None
peakCount = int(df.loc[peak_idx, "Count"]) if peak_idx is not None else 0

# ---- Save stats to CSV (single row) ----
stats_df = pd.DataFrame([{
    "total": total,
    "avgPerYear": avgPerYear,
    "peakYear": peakYear,
    "peakCount": peakCount
}])

stats_df.to_csv(OUT_STATS_CSV, index=False)
stats_df


,total,avgPerYear,peakYear,peakCount
0,3530,101,2020,157


### papersByConferenceData

In [14]:
df = pd.read_csv(src_path)

df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df = df.dropna(subset=["Year"]).copy()
df["Year"] = df["Year"].astype(int)

# mapping minimale
MAP = {"Vis": "vis", "InfoVis": "infovis", "VAST": "vast", "SciVis": "scivis"}
df["conf_key"] = df["Conference"].map(MAP)

wide = (
    df.dropna(subset=["conf_key"])
      .groupby(["Year", "conf_key"]).size()
      .unstack(fill_value=0)
      .reset_index()
      .sort_values("Year")
)

# assicura colonne
for k in ["vis","infovis","vast","scivis"]:
    if k not in wide.columns:
        wide[k] = 0

wide = wide[["Year","vis","infovis","vast","scivis"]]
wide.to_csv("../site/data/papers/papersByConference.csv", index=False)

records = [
    {"year": int(r.Year), "vis": int(r.vis), "infovis": int(r.infovis), "vast": int(r.vast), "scivis": int(r.scivis)}
    for r in wide.itertuples(index=False)
]

js = (
"/** AUTO-GENERATED */\n"
f"export const papersByConferenceData = {json.dumps(records, indent=2)};\n\n"
f"export const conferenceKeys = {json.dumps(['vis','infovis','vast','scivis'])};\n\n"
f"export const conferenceLabels = {json.dumps({'vis':'Vis','infovis':'InfoVis','vast':'VAST','scivis':'SciVis'}, indent=2)};\n"
)
open("../site/data/papers/papersByConferenceData.js","w",encoding="utf-8").write(js)


3479

### papersByPublicationData

In [15]:
df = pd.read_csv(src_path)

df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
df = df.dropna(subset=["Year"]).copy()
df["Year"] = df["Year"].astype(int)

# mapping minimale
MAP = {"J": "journal", "C": "conference"}
df["type_key"] = df["PaperType"].map(MAP)

wide = (
    df.dropna(subset=["type_key"])
      .groupby(["Year", "type_key"]).size()
      .unstack(fill_value=0)
      .reset_index()
      .sort_values("Year")
)

# assicura colonne
for k in ["journal","conference"]:
    if k not in wide.columns:
        wide[k] = 0

wide = wide[["Year","journal","conference"]]
wide.to_csv("../site/data/papers/papersByPublication.csv", index=False)

records = [
    {"year": int(r.Year), "journal": int(r.journal), "conference": int(r.conference)}
    for r in wide.itertuples(index=False)
]

js = (
"/** AUTO-GENERATED */\n"
f"export const papersByPublicationData = {json.dumps(records, indent=2)};\n\n"
f"export const publicationKeys = {json.dumps(['journal','conference'])};\n\n"
f"export const publicationLabels = {json.dumps({'journal':'Journal (TVCG)','conference':'Conference'}, indent=2)};\n"
)
open("../site/data/papers/papersByPublicationData.js","w",encoding="utf-8").write(js)

2559

### topicsTreemapData

In [16]:
df = pd.read_csv(src_path)

# --- normalizza colonne (Topic può essere Cluster)
topic_col = "Topic" if "Topic" in df.columns else ("Cluster" if "Cluster" in df.columns else None)
if topic_col is None:
    raise ValueError("Missing Topic/Cluster column in dataset_clean")

macro_col = "MacroCategory" if "MacroCategory" in df.columns else None
if macro_col is None:
    raise ValueError("Missing MacroCategory column in dataset_clean")

name_col = "CategoryShort" if "CategoryShort" in df.columns else ("ClusterShort" if "ClusterShort" in df.columns else None)

df["TopicId"] = pd.to_numeric(df[topic_col], errors="coerce").astype("Int64")
df = df.dropna(subset=["TopicId", macro_col]).copy()
df["TopicId"] = df["TopicId"].astype(int)

# se manca il nome short, crea placeholder
if name_col is None:
    df["TopicShort"] = df["TopicId"].map(lambda x: f"Topic {int(x)}")
else:
    df["TopicShort"] = df[name_col].fillna(df["TopicId"].map(lambda x: f"Topic {int(x)}"))

df["MacroCategory"] = df[macro_col].astype(str)

# --- 1) macro -> numero di topic unici (e papers totali)
macro_counts = (
    df.groupby("MacroCategory")
      .agg(topics=("TopicId", "nunique"), papers=("TopicId", "size"))
      .reset_index()
      .sort_values(["topics","papers"], ascending=False)
)

macro_counts.to_csv("../site/data/papers/macroTopicsCounts.csv", index=False)

# --- 2) dettaglio macro -> topic -> papers
detail = (
    df.groupby(["MacroCategory", "TopicId", "TopicShort"])
      .size()
      .reset_index(name="papers")
      .sort_values(["MacroCategory", "papers"], ascending=[True, False])
)

detail.to_csv("../site/data/papers/macroTopicsDetail.csv", index=False)

# --- treemap structure (MacroCategory as group, TopicShort as leaves)
treemap = {"name": "Topics", "children": []}

for macro_name, sub in detail.groupby("MacroCategory"):
    children = [
        {"name": r.TopicShort, "value": int(r.papers), "topic_id": int(r.TopicId)}
        for r in sub.itertuples(index=False)
    ]
    treemap["children"].append({"name": macro_name, "children": children})

# --- JS output
js = (
    "/** AUTO-GENERATED */\n"
    f"export const topicsTreemapData = {json.dumps(treemap, ensure_ascii=False, indent=2)};\n\n"
    "export const topicColors = {\n"
    '  "Meshes & Volume Rendering": "#1E88E5",\n'
    '  "Flow Fields & CFD Visualization": "#00897B",\n'
    '  "Dimensionality Reduction & Multivariate Plots": "#8E24AA",\n'
    '  "Surgical Planning & Tomography": "#D32F2F",\n'
    '  "Diffusion MRI & Tractography": "#3949AB",\n'
    '  "VIS Literature & Bibliometrics": "#6D4C41",\n'
    '  "Dashboards & Infographics": "#F57C00",\n'
    '  "Twitter & Social Media": "#C2185B",\n'
    '  "Time Series & Temporal Patterns": "#388E3C",\n'
    '  "Ultrasound Volume Rendering & Segmentation": "#455A64"\n'
    "};\n"
)

open("../site/data/papers/topicsTreemapData.js", "w", encoding="utf-8").write(js)


5009

### citationsHistogramData

In [17]:
df = pd.read_csv(src_path)

# --- pick + clean numeric columns
df["cit_crossref"] = pd.to_numeric(df.get("CitationCount_CrossRef"), errors="coerce")
df["cit_aminer"] = pd.to_numeric(df.get("AminerCitationCount"), errors="coerce")

# --- Best-of merge: max(CrossRef, Aminer) if both exist; else whichever exists; else 0
c1 = df["cit_crossref"]
c2 = df["cit_aminer"]

cit = c1.copy()
cit[cit.isna()] = c2[cit.isna()]

both = c1.notna() & c2.notna()
cit[both] = np.maximum(c1[both], c2[both])

df["Citations"] = cit.fillna(0)
df.loc[df["Citations"] < 0, "Citations"] = 0
df["Citations"] = df["Citations"].round(0).astype(int)

# --- histogram raw data (one value per paper)
records = df["Citations"].tolist()

# --- stats
vals = np.array(records, dtype=int)
median = int(np.median(vals)) if len(vals) else 0
mean = float(vals.mean()) if len(vals) else 0.0
maxv = int(vals.max()) if len(vals) else 0
papers100 = int((vals >= 100).sum())
papers500 = int((vals >= 500).sum())

# Coverage (optional but useful to debug)
cov_crossref = int(df["cit_crossref"].notna().sum())
cov_aminer = int(df["cit_aminer"].notna().sum())

citationStats = {
    "median": median,
    "mean": round(mean, 1),
    "max": maxv,
    "papersWith100Plus": papers100,
    "papersWith500Plus": papers500,
    "crossrefCoverage": cov_crossref,
    "aminerCoverage": cov_aminer,
}

# --- bins (keep your mock bins, but ensure last threshold > max)
thresholds = [0, 5, 10, 20, 35, 50, 75, 100, 150, 200, 500, 1000,1500,4000]
last = max(4000, ((maxv // 500) + 1) * 500)  # ensures > maxv
if last <= thresholds[-1]:
    last = thresholds[-1] * 3
thresholds = thresholds + [last]

labels = []
for i in range(len(thresholds) - 2):
    labels.append(f"{thresholds[i]}-{thresholds[i+1]}")
labels.append(f"{thresholds[-2]}+")  # open-ended label like "1000+"

histogramBins = {
    "thresholds": thresholds,
    "labels": labels
}

# --- JS output
js = (
    "/** AUTO-GENERATED */\n"
    "/* Citation counts derived as max(CitationCount_CrossRef, AminerCitationCount) with fallback to available source */\n\n"
    f"export const citationsHistogramData = {json.dumps(records, indent=2)};\n\n"
    f"export const citationStats = {json.dumps(citationStats, indent=2)};\n\n"
    f"export const histogramBins = {json.dumps(histogramBins, indent=2)};\n"
)

out_path = "../site/data/papers/citationsHistogramData.js"
open(out_path, "w", encoding="utf-8").write(js)
print("Wrote:", out_path)


Wrote: ../site/data/papers/citationsHistogramData.js


In [18]:
# --- OUTPUT ---
out_csv = Path("../site/data/papers/awardsByType.csv")
out_js  = Path("../site/data/papers/awardsData.js")

# --- Load ---
df = pd.read_csv(src_path)

# --- Find award column (robusto) ---
candidate_cols = ["Award", "Awards", "AwardCode", "AwardCodes", "VISAward", "InternalAward"]
award_col = next((c for c in candidate_cols if c in df.columns), None)
if award_col is None:
    raise ValueError(f"Nessuna colonna award trovata. Cercate: {candidate_cols}. Colonne disponibili: {list(df.columns)[:30]} ...")

# --- Parse award codes ---
VALID = {"BP","HM","TT","BA","BCS"}

def parse_awards(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s or s.lower() in {"nan","none","null"}:
        return []

    # list-like string? e.g. "['BP','HM']"
    if s.startswith("[") and s.endswith("]"):
        try:
            lst = ast.literal_eval(s)
            if isinstance(lst, (list, tuple)):
                parts = [str(v).strip() for v in lst if str(v).strip()]
            else:
                parts = [s]
        except Exception:
            parts = [s]
    else:
        parts = [s]

    out = []
    for part in parts:
        # split su separatori comuni: ; , | / spazio e anche "+"
        tokens = re.split(r"[;,|/]+|\s+|\+", str(part))
        tokens = [t.strip().upper() for t in tokens if t.strip()]
        out.extend(tokens)

    # tieni solo codici validi
    out = [t for t in out if t in VALID]
    return out

award_lists = df[award_col].apply(parse_awards)

# --- Counts (ogni codice conta, anche se un paper ha più codici) ---
flat = [a for lst in award_lists for a in lst]
counts = pd.Series(flat).value_counts()

# assicurati che tutti i codici ci siano (anche a zero)
counts = counts.reindex(["HM","BP","TT","BA","BCS"]).fillna(0).astype(int)

total_awards = int(counts.sum())
pct = (counts / total_awards * 100).round(2) if total_awards > 0 else counts.astype(float)

df_out = pd.DataFrame({
    "Award": counts.index,
    "Count": counts.values,
    "Percentage": pct.values
})

out_csv.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_csv, index=False)
print("Wrote:", out_csv.resolve())
display(df_out)

# --- Stats per UI ---
papers_total = int(len(df))
papers_awarded = int((award_lists.apply(len) > 0).sum())
percentage_awarded = round((papers_awarded / papers_total) * 100, 2) if papers_total else 0.0

awardStats = {
    "total": total_awards,
    "percentageAwarded": percentage_awarded,     # % papers con almeno 1 award
    "papersAwarded": papers_awarded,
    "papersTotal": papers_total
}

# --- JS awardsData (compatibile col tuo chart: usa d.type) ---
LABELS = {
    "BP": {"type": "Best Paper", "icon": "🏆"},
    "HM": {"type": "Honorable Mention", "icon": "🎖️"},
    "TT": {"type": "Test of Time", "icon": "⏰"},
    "BA": {"type": "Best Application Paper", "icon": "🧩"},
    "BCS": {"type": "Best Case Study", "icon": "📊"},
}

awardsData = []
for code in ["BP","HM","BCS","TT","BA"]:  # ordine “carino” per il grafico
    c = int(counts.get(code, 0))
    meta = LABELS[code]
    awardsData.append({
        "type": meta["type"],
        "count": c,
        "icon": meta["icon"],
        "code": code,   # extra: non rompe il chart
    })

pictogramCellValue = 5  # puoi cambiare a 10 se vuoi meno quadratini

js = (
"/** AUTO-GENERATED */\n"
"// Award counts by type for pictogram bar chart\n\n"
f"export const awardsData = {json.dumps(awardsData, ensure_ascii=False, indent=2)};\n\n"
f"export const awardStats = {json.dumps(awardStats, ensure_ascii=False, indent=2)};\n\n"
f"export const pictogramCellValue = {int(pictogramCellValue)};\n"
)

out_js.write_text(js, encoding="utf-8")
print("Wrote:", out_js.resolve())


Wrote: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/site/data/papers/awardsByType.csv


,Award,Count,Percentage
0,HM,144,52.17
1,BP,83,30.07
2,TT,38,13.77
3,BA,6,2.17
4,BCS,5,1.81


Wrote: /Users/irynasavchuk/Desktop/DATAVIZ_PROJECT/DV/dv_repo/site/data/papers/awardsData.js
